## Assignment: Scaling Laws, Supervised Fine-Tuning, and Parameter-Efficient Adaptation
### Course: Advanced NLP

# Part 1: Investigating Scaling Laws (60 points — mandatory)


# Part 2 Supervised Fine-Tuning (SFT) (20 points — optional)

We fine-tune the pretrained baseline (config M, 100% Shakespeare) on two structured tasks extracted from the Shakespeare corpus:
- **Task A**: Speaker Identification: predict which character speaks a given line
- **Task B**: Verse vs. Prose Classification: classify a passage as verse or prose

All experiments start from `out-shakespeare-baseline/ckpt.pt`.

In [ ]:
# Task A: Speaker Identification, Task B: Verse vs. Prose Classification, combined
!python data/shakespeare_char_sft_A/prepare_finetune_speakers.py --task A
!python data/shakespeare_char_sft_B/prepare_finetune_speakers.py --task B
!python data/shakespeare_char_sft_combined/prepare_finetune_speakers.py 

ERROR: unknown command "python3"
ERROR: unknown command "python3"
ERROR: unknown command "python3"


### Experiment 1: Single-Task SFT
Fine-tune separate models on Task A and Task B independently.

In [ ]:
!python train.py config/finetune_shakespeare_sft_A.py | tee training_logs/sft_A.log
!python train.py config/finetune_shakespeare_sft_B.py | tee training_logs/sft_B.log

### Experiment 2: Multi-task SFT
Fine-tune the combined model on Task A and Task B independently.

In [ ]:
!python train.py config/finetune_shakespeare_sft_combined.py | tee training_logs/sft_combined.log

### Report Accuracy on Validation Sets
`evaluate_sft.py` reports exact-match accuracy: the predicted token sequence between `[ANSWER]` and `[END]` must match the ground truth exactly.

Report the accuracy on both Task A and Task B test sets for each model
- SFT A model 
- SFT B model
- Combined model
- Baseline model

In [ ]:
# Task A sft A
!python evaluate_sft.py --task A \
    --checkpoint out-shakespeare-sft_A/ckpt.pt \
    --data_dir data/shakespeare_char_sft_A | tee evaluation_logs/sft_A_for_task_A.log

# Task B sft B
!python evaluate_sft.py --task B \
    --checkpoint out-shakespeare-sft_B/ckpt.pt \
    --data_dir data/shakespeare_char_sft_B | tee evaluation_logs/sft_B_for_task_B.log

# Task A combined
!python evaluate_sft.py --task A \
    --checkpoint out-shakespeare-combined/ckpt.pt \
    --data_dir data/shakespeare_char_sft_A | tee evaluation_logs/sft_combined_for_task_A.log

# Task B combined
!python evaluate_sft.py --task B \
    --checkpoint out-shakespeare-combined/ckpt.pt \
    --data_dir data/shakespeare_char_sft_B | tee evaluation_logs/sft_combined_for_task_B.log

# Task A baseline
!python evaluate_sft.py --task A \
    --checkpoint out-shakespeare-baseline/ckpt.pt \
    --data_dir data/shakespeare_char_sft_A | tee evaluation_logs/baseline_for_task_A.log

# Task B baseline
!python evaluate_sft.py --task B \
    --checkpoint out-shakespeare-baseline/ckpt.pt \
    --data_dir data/shakespeare_char_sft_B | tee evaluation_logs/baseline_for_task_B.log

# Cross-task evaluation: measure how each single-task model performs on the OTHER task
# (serves as a reference point for the multi-task vs. single-task comparison)

# Task A sft B
!python evaluate_sft.py --task A \
    --checkpoint out-shakespeare-sft_B/ckpt.pt \
    --data_dir data/shakespeare_char_sft_A | tee evaluation_logs/sft_B_for_task_A.log

# Task B sft A
!python evaluate_sft.py --task B \
    --checkpoint out-shakespeare-sft_A/ckpt.pt \
    --data_dir data/shakespeare_char_sft_B | tee evaluation_logs/sft_A_for_task_B.log

### Experiment 3: Catastrophic Forgetting
Measure whether SFT degrades general Shakespeare language modelling ability by comparing validation loss before and after fine-tuning.
In two ways: 
(1) generate unconditional samples to compare fluency, 
(2) compute validation loss on the original Shakespeare set before and after SFT.

In [ ]:
!python sample.py --out_dir=out-shakespeare-baseline \ --device=cpu --num_samples=3 --max_new_tokens=500
!python sample.py --out_dir=out-shakespeare-sft_A \ --device=cpu --num_samples=3 --max_new_tokens=500
!python sample.py --out_dir=out-shakespeare-sft_B \ --device=cpu --num_samples=3 --max_new_tokens=500
!python sample.py --out_dir=out-shakespeare-combined \ --device=cpu --num_samples=3 --max_new_tokens=500

In [ ]:
!python catastrophic-forgetting.py | tee evaluation_logs/catastrophic-forgetting.log

### Plot all the results
- training curves
- accuracy comparison
- catastrophic forgetting
- results table

In [ ]:
!python plot_exp_2.py 

# Part 3: Low-Rank Adaptation — LoRA (20 points — optional)

### Experiment 4: LoRA single-task SFT

Implement LoRA in nanoGPT in lora_train.py

Fine-tune with LoRA (rank 4) on Task A.

The dataset_name, lora_rank, and task_name depend on the experiment.


In [ ]:
!python lora_train.py  
# dataset = 'shakespeare_char_sft_A' , 
# lora_rank = 4, 
#task_name = "A"

Fine-tune with LoRA (rank 4) on Task B.

In [ ]:
!python lora_train.py  
# dataset = 'shakespeare_char_sft_B' , 
# lora_rank = 4, 
#task_name = "B"

In [ ]:
!python evaluate_ranks.py

#data_dir = "data/shakespeare_char_sft_A"
#ckpt_dir = "out"
#checkpoints = ["ckpt_A.pt",]


In [ ]:
!python evaluate_ranks.py

#data_dir = "data/shakespeare_char_sft_B"
#ckpt_dir = "out"
#checkpoints = ["ckpt_B.pt",]


### Experiment 5: LoRA rank ablation

In [ ]:
!python lora_train.py  
# dataset = 'shakespeare_char_sft_A' , 
# lora_rank = 1, 
#task_name = "A_1"

In [ ]:
!python lora_train.py  
# dataset = 'shakespeare_char_sft_A' , 
# lora_rank = 2, 
#task_name = "A_2"

In [ ]:
!python lora_train.py  
# dataset = 'shakespeare_char_sft_A' , 
# lora_rank = 4, 
#task_name = "A_4"

In [ ]:
!python lora_train.py  
# dataset = 'shakespeare_char_sft_A' , 
# lora_rank = 8, 
#task_name = "A_8"

In [ ]:
!python lora_train.py  
# dataset = 'shakespeare_char_sft_A' , 
# lora_rank = 16, 
#task_name = "A_16"

In [ ]:
!python evaluate_ranks.py

#data_dir = "data/shakespeare_char_sft_A"
#ckpt_dir = "out"
#checkpoints = [ "ckpt_A_r1.pt","ckpt_A_r2.pt","ckpt_A_r4.pt","ckpt_A_r8.pt","ckpt_A_r16.pt",]


Plot fine-tuning and LoRA at each rank.

In [ ]:
!python exp5_lora_rank_ablation.py